# MNIST Dataset with Logistic Regression and ANN

## 1. Libraries and dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score

# Logistic Reg
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression

# ANN
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.utils import to_categorical

Reading data

In [ ]:
train = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

## 2. EDA

In [ ]:
train.info()

In [ ]:
train.head()

In [ ]:
test.info()

In [ ]:
test.head()

In [ ]:
# features from train table
X = train.drop("label", axis=1)


# target from train table
y = train["label"]

Checking data imbalance

In [ ]:


plt.figure(figsize=(8, 4))
sns.countplot(x=y, color = 'steelblue')
plt.title("Class distribution in training data")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

y.value_counts()

# it seems that target class is well-balanced

Pixel-value is an integer between 0 and 255 - so I divide this value by 255 to get values between 0 and 1 - it should be lighter for computation

In [ ]:
X = X/255
test = test/255

Changing labels to one hot vectors - necessary to use categorical_crossentropy loss function

In [ ]:
y_cat = to_categorical(y, num_classes = 10)

Train and test subsets for Logistic Reg

In [ ]:
X_train_LR, X_test_LR, y_train_LR, y_test_LR = train_test_split(
    X, y, test_size= 0.2, random_state=42)

Train and validation subsets for ANN

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size= 0.2, random_state=42)

## 3. Logistic Regression with k-fold CV (WORK IN PROGRESS)

In [ ]:
LogReg = LogisticRegression(
    multi_class="multinomial", # multinomial because of 10 classes
    solver="saga", # saga solver shoud be efficient for big data
    max_iter=500, # 200 not enough, 500 is taking too much time
    n_jobs=-1 # efficient usage of CPU
)

Defining k-fold CV. Stratified KFold due to sampling with class proportion

In [ ]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Perform CV with multiple metrics (macro because data are well-balanced)

In [ ]:
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

Training model

In [ ]:
# Training is taking too much time
# results_LR = cross_validate(LogReg, X_train_LR, y_train_LR, cv=kfold, scoring=scoring)

Results of accuracy, precision, recall, f1: mean, min, max

results = {
    "Logistic Regression": {
        "Accuracy": (results_LR['test_accuracy'].mean(),
                     results_LR['test_accuracy'].min(),
                     results_LR['test_accuracy'].max()),
        "Precision": (results_LR['test_precision_macro'].mean(),
                      results_LR['test_precision_macro'].min(),
                      results_LR['test_precision_macro'].max()),
        "Recall": (results_LR['test_recall_macro'].mean(),
                   results_LR['test_recall_macro'].min(),
                   results_LR['test_recall_macro'].max()),
        "F1": (results_LR['test_f1_macro'].mean(),
               results_LR['test_f1_macro'].min(),
               results_LR['test_f1_macro'].max())
    }
}

Split mean, min, max into columns
df_results = pd.DataFrame(results)
df_mean = df_results.map(lambda x: x[0])
df_min = df_results.map(lambda x: x[1])
df_max = df_results.map(lambda x: x[2])

print("Mean values:\n", df_mean)
print("\nMin values:\n", df_min)
print("\nMax values:\n", df_max)

Compute lower and upper errors
yerr_lower = df_mean - df_min
yerr_upper = df_max - df_mean

Stack into (n_metrics, 2, n_models) - making sure yerr has the exact shape matplotlib needs:
yerr = np.array([yerr_lower.to_numpy(), yerr_upper.to_numpy()]).transpose(2,0,1)

Plot
ax = df_mean_t.plot(
    kind="bar",
    figsize=(10,6),
    yerr=yerr,
    capsize=4
)

plt.title("Model Comparison (5-fold CV)")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend(title="Model", loc="lower right")
plt.show()

Printing metrics

Metrics on a graph

## 4. ANN model

Creating ANN model

In [ ]:
ANN_model = Sequential()

In [ ]:
ANN_model.add(Dense(128,activation='relu'))

ANN_model.add(Dense(64,activation='relu'))

ANN_model.add(Dense(32,activation='relu'))

ANN_model.add(Dense(10,activation='softmax'))

# For a multi-class classification problem loss function is categorical_crossentropy
ANN_model.compile(loss='categorical_crossentropy', optimizer='adam')

Model training

In [ ]:
ANN_model.fit(x=X_train, 
          y=y_train, 
          epochs=10,
          validation_data=(X_val, y_val), verbose=2
          )

Loss function for training set and validation set - is there an overfitting problem? In this case validation loss starts increasing quite fast

In [ ]:
ANN_model_loss = pd.DataFrame(ANN_model.history.history)
ANN_model_loss.plot()

Model evaluation

In [ ]:
y_val_pred = ANN_model.predict(X_val).argmax(axis=1)
y_val_true = y_val.argmax(axis=1)

In [ ]:
print(classification_report(y_val_true,y_val_pred))

In [ ]:
cm = confusion_matrix(y_val_true, y_val_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - ANN on MNIST")
plt.show()

Adding a dropout in order to reduce overfitting

In [ ]:
from tensorflow.keras.layers import Dropout

In [ ]:
ANN_model_d = Sequential()

ANN_model_d.add(Dense(128,activation='relu'))
ANN_model_d.add(Dropout(0.2))

ANN_model_d.add(Dense(64,activation='relu'))
ANN_model_d.add(Dropout(0.2))

ANN_model_d.add(Dense(32,activation='relu'))
ANN_model_d.add(Dropout(0.2))

ANN_model_d.add(Dense(10,activation='softmax'))

# For a multi-class classification problem loss function is categorical_crossentropy
ANN_model_d.compile(loss='categorical_crossentropy', optimizer='adam')

In [ ]:
ANN_model_d.fit(x=X_train, 
          y=y_train, 
          epochs=10,
          validation_data=(X_val, y_val), verbose=2
          )

In [ ]:
ANN_model_loss = pd.DataFrame(ANN_model_d.history.history)
ANN_model_loss.plot()

In [ ]:
y_val_pred_2 = ANN_model_d.predict(X_val).argmax(axis=1)
y_val_true_2 = y_val.argmax(axis=1)

In [ ]:
print(classification_report(y_val_true_2,y_val_pred_2))

In [ ]:
cm2 = confusion_matrix(y_val_true_2, y_val_pred_2)
plt.figure(figsize=(8, 6))
sns.heatmap(cm2, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - ANN on MNIST")
plt.show()

## 5. Summary
* ANN with droput has better accuracy, recall and precision.All those metrics have 0.97 score
* ANN without droput has metrics with 0.96 score and it seems to have an overfitting issue
* If training had more epochs, I would use early stopping to prevent overfitting at some point